A quí se desglosa paso a paso cómo cumplir con los requisitos del **subject**, enfocándonos en los módulos de **cybersecurity**.  Implementación de cada uno de los módulos, incluyendo herramientas, configuraciones y buenas prácticas.

---

## **1. Major Module: WAF/ModSecurity y HashiCorp Vault**

### **Paso 1: Implementar un WAF (Web Application Firewall) con ModSecurity**

#### **¿Qué es ModSecurity?**
ModSecurity es un WAF de código abierto que protege aplicaciones web contra ataques comunes, como inyecciones SQL, XSS, y otros.

#### **Pasos para implementar ModSecurity:**

1. **Instalar ModSecurity**:
   - Si estás usando **Nginx** o **Apache**, instala ModSecurity como un módulo.
   - Para Nginx:
     ```bash
     sudo apt install libnginx-mod-security
     ```
   - Para Apache:
     ```bash
     sudo apt install libapache2-mod-security2
     ```

2. **Configurar ModSecurity**:
   - Descarga las reglas base de OWASP para ModSecurity:
     ```bash
     git clone https://github.com/SpiderLabs/owasp-modsecurity-crs.git
     ```
   - Copia las reglas a la configuración de ModSecurity:
     ```bash
     sudo cp -r owasp-modsecurity-crs/rules/ /etc/modsecurity/
     sudo cp owasp-modsecurity-crs/crs-setup.conf.example /etc/modsecurity/crs-setup.conf
     ```

3. **Habilitar ModSecurity**:
   - En Nginx:
     - Edita el archivo de configuración de Nginx (`/etc/nginx/nginx.conf`) y agrega:
       ```nginx
       load_module modules/ngx_http_modsecurity_module.so;
       modsecurity on;
       modsecurity_rules_file /etc/modsecurity/modsecurity.conf;
       ```
   - En Apache:
     - Habilita el módulo:
       ```bash
       sudo a2enmod security2
       ```
     - Edita el archivo de configuración de Apache (`/etc/apache2/mods-enabled/security2.conf`) y agrega:
       ```apache
       <IfModule security2_module>
           SecRuleEngine On
           Include /etc/modsecurity/crs-setup.conf
           Include /etc/modsecurity/rules/*.conf
       </IfModule>
       ```

4. **Reiniciar el servidor**:
   - Para Nginx:
     ```bash
     sudo systemctl restart nginx
     ```
   - Para Apache:
     ```bash
     sudo systemctl restart apache2
     ```

5. **Probar el WAF**:
   - Realiza pruebas de penetración con herramientas como **OWASP ZAP** o **Burp Suite** para asegurarte de que el WAF esté bloqueando ataques.

---

### **Paso 2: Implementar HashiCorp Vault para la gestión de secretos**

#### **¿Qué es HashiCorp Vault?**
HashiCorp Vault es una herramienta para almacenar y gestionar secretos de manera segura, como API keys, contraseñas y certificados.

#### **Pasos para implementar HashiCorp Vault:**

1. **Instalar Vault**:
   - Descarga e instala Vault desde el sitio oficial:
     ```bash
     wget https://releases.hashicorp.com/vault/1.13.0/vault_1.13.0_linux_amd64.zip
     unzip vault_1.13.0_linux_amd64.zip
     sudo mv vault /usr/local/bin/
     ```

2. **Iniciar Vault en modo desarrollo**:
   - Ejecuta Vault en modo desarrollo (solo para pruebas):
     ```bash
     vault server -dev
     ```
   - Configura la variable de entorno `VAULT_ADDR`:
     ```bash
     export VAULT_ADDR='http://127.0.0.1:8200'
     ```

3. **Almacenar secretos en Vault**:
   - Escribe un secreto en Vault:
     ```bash
     vault kv put secret/myapp api_key=my_secret_key
     ```
   - Lee un secreto desde Vault:
     ```bash
     vault kv get secret/myapp
     ```

4. **Integrar Vault con tu aplicación**:
   - Usa el cliente de Vault en tu aplicación para obtener secretos en tiempo de ejecución.
   - Por ejemplo, en Python:
     ```python
     import hvac
     client = hvac.Client(url='http://127.0.0.1:8200')
     secret = client.read('secret/data/myapp')
     api_key = secret['data']['data']['api_key']
     ```

---

## **2. Minor Module: GDPR Compliance**

### **Paso 3: Implementar GDPR Compliance**

#### **Funcionalidades requeridas:**
1. **Anonimización de datos**.
2. **Gestión de datos locales**.
3. **Eliminación de cuentas**.

#### **Pasos para implementar GDPR Compliance:**

1. **Anonimización de datos**:
   - Crea una función que reemplace los datos personales de un usuario con valores anónimos.
   - Ejemplo en Python:
     ```python
     def anonymize_user(user):
         user.name = "Anonymous"
         user.email = f"anonymous{user.id}@example.com"
         user.save()
     ```

2. **Gestión de datos locales**:
   - Proporciona una interfaz para que los usuarios puedan:
     - Ver sus datos.
     - Editar sus datos.
     - Solicitar la eliminación de sus datos.
   - Ejemplo en Django:
     ```python
     from django.contrib.auth.decorators import login_required
     from django.shortcuts import render, redirect

     @login_required
     def view_data(request):
         return render(request, 'view_data.html', {'user': request.user})

     @login_required
     def delete_account(request):
         if request.method == 'POST':
             request.user.delete()
             return redirect('home')
         return render(request, 'delete_account.html')
     ```

3. **Eliminación de cuentas**:
   - Implementa un proceso para eliminar cuentas y todos los datos asociados.
   - Asegúrate de que los datos se eliminen de manera segura y permanente.

---

## **3. Major Module: 2FA y JWT**

### **Paso 4: Implementar Two-Factor Authentication (2FA)**

#### **Pasos para implementar 2FA:**

1. **Usar una librería para 2FA**:
   - En Python, puedes usar `pyotp` para generar códigos OTP:
     ```bash
     pip install pyotp
     ```

2. **Generar y verificar códigos OTP**:
   - Genera un código OTP:
     ```python
     import pyotp
     totp = pyotp.TOTP("base32secret3232")
     otp_code = totp.now()
     ```
   - Verifica el código OTP:
     ```python
     is_valid = totp.verify(otp_code)
     ```

3. **Integrar 2FA en tu aplicación**:
   - Proporciona una opción para que los usuarios habiliten 2FA en su perfil.
   - Guarda el secreto de 2FA en la base de datos (asegúrate de cifrarlo).

---

### **Paso 5: Implementar JSON Web Tokens (JWT)**

#### **Pasos para implementar JWT:**

1. **Usar una librería para JWT**:
   - En Python, puedes usar `PyJWT`:
     ```bash
     pip install pyjwt
     ```

2. **Generar y verificar tokens JWT**:
   - Genera un token JWT:
     ```python
     import jwt
     token = jwt.encode({'user_id': 1}, 'secret', algorithm='HS256')
     ```
   - Verifica un token JWT:
     ```python
     try:
         payload = jwt.decode(token, 'secret', algorithms=['HS256'])
         user_id = payload['user_id']
     except jwt.InvalidTokenError:
         print("Token inválido")
     ```

3. **Proteger rutas con JWT**:
   - Usa un middleware para verificar el token JWT en cada solicitud.

---

## **4. Pruebas y Documentación**

### **Paso 6: Pruebas de seguridad**
- Usa **OWASP ZAP** para realizar pruebas de penetración.
- Verifica que todas las funcionalidades de seguridad estén funcionando correctamente.

### **Paso 7: Documentación**
- Documenta cómo has implementado cada módulo de seguridad.
- Incluye capturas de pantalla y ejemplos de uso.

---

### **Resumen**

1. **WAF/ModSecurity**: Protege tu aplicación contra ataques web.
2. **HashiCorp Vault**: Gestiona secretos de manera segura.
3. **GDPR Compliance**: Implementa anonimización, gestión de datos y eliminación de cuentas.
4. **2FA y JWT**: Mejora la autenticación y autorización.

¡Sigue estos pasos y cumplirás con el **subject** de tu proyecto! Si tienes más preguntas, no dudes en preguntar. ¡Buena suerte! 🚀

## **Resumen de lo realizado**

### **1. Despliegue de servicios de seguridad**
Hemos configurado y desplegado los siguientes servicios de seguridad en tu entorno Docker:

1. **OWASP ZAP (Zed Attack Proxy)**:
   - **Función**: Herramienta de escaneo de seguridad para aplicaciones web. Detecta vulnerabilidades como XSS, SQL Injection, CSRF, etc.
   - **Puerto**: 8081.
   - **Configuración**:
     - Se inició en modo daemon (`-daemon`).
     - Se expuso el puerto 8081 para acceder desde el host.
     - Se configuró una API Key para interactuar con ZAP programáticamente.

2. **HashiCorp Vault**:
   - **Función**: Herramienta para gestionar secretos y datos sensibles de manera segura (API keys, contraseñas, certificados, etc.).
   - **Puerto**: 8200.
   - **Configuración**:
     - Se inició en modo desarrollo (`-dev`).
     - Se configuró para escuchar en `0.0.0.0` para que sea accesible desde fuera del contenedor.
     - Se almacenó un secreto de ejemplo (`api_key=my_secret_key`) en Vault.

3. **ModSecurity**:
   - **Función**: Firewall de aplicaciones web (WAF) que protege contra ataques comunes como inyecciones SQL, XSS, etc.
   - **Configuración**:
     - Se instaló y configuró como módulo de Apache.
     - Se copió un archivo de configuración básico (`modsecurity.conf`) para habilitar el motor de reglas.

4. **Apache HTTP Server**:
   - **Función**: Servidor web que aloja la aplicación y se integra con ModSecurity para protección adicional.
   - **Puerto**: 80 (interno) y 8080 (expuesto en el host).

---

### **2. Errores solucionados**

1. **Vault no accesible desde el host**:
   - **Problema**: Vault estaba configurado para escuchar en `127.0.0.1`, lo que impedía el acceso desde fuera del contenedor.
   - **Solución**: Se modificó el comando de inicio de Vault para que escuche en `0.0.0.0`:
     ```bash
     vault server -dev -dev-listen-address="0.0.0.0:8200"
     ```

2. **Volúmenes no persistentes**:
   - **Problema**: Los datos de ZAP y ModSecurity no se guardaban en los volúmenes Docker.
   - **Solución**:
     - Se verificaron los permisos y rutas de los volúmenes en `docker-compose.yml`.
     - Se aseguró que los directorios montados existieran y tuvieran permisos de escritura.

3. **ZAP no generaba reportes**:
   - **Problema**: El script `zap_scan.sh` no guardaba el reporte en el volumen correcto.
   - **Solución**: Se verificó la ruta de salida del reporte y se copió manualmente el archivo `zap_report.html` al host si era necesario.

4. **Integración de servicios**:
   - **Problema**: Los servicios no estaban correctamente integrados (por ejemplo, el backend no podía acceder a Vault).
   - **Solución**: Se configuró la variable de entorno `VAULT_ADDR` en el servicio `backend` para que apunte a `http://security:8200`.

---

### **3. Función de cada servicio de seguridad**

1. **OWASP ZAP**:
   - Escanea la aplicación en busca de vulnerabilidades comunes.
   - Genera reportes detallados en formato HTML.
   - Permite realizar pruebas manuales y automatizadas.

2. **HashiCorp Vault**:
   - Almacena y gestiona secretos de manera segura.
   - Proporciona una API para acceder a los secretos desde otros servicios (por ejemplo, el backend).

3. **ModSecurity**:
   - Protege la aplicación contra ataques web comunes.
   - Registra intentos de ataques en logs para su posterior análisis.

4. **Apache HTTP Server**:
   - Sirve la aplicación web.
   - Se integra con ModSecurity para añadir una capa adicional de seguridad.

---

## **Lista de pruebas que puedes realizar**

### **1. Pruebas de seguridad con OWASP ZAP**
- **Escaneo activo**:
  - Ejecuta un escaneo activo contra la aplicación para detectar vulnerabilidades.
  - Comando: `docker exec -it security /zap/wrk/zap_scan.sh`.
- **Revisar reportes**:
  - Abre el archivo `zap_report.html` generado por ZAP para ver los resultados del escaneo.
- **Pruebas manuales**:
  - Usa la interfaz web de ZAP (disponible en `http://localhost:8081`) para realizar pruebas manuales.

### **2. Pruebas con HashiCorp Vault**
- **Acceder a secretos**:
  - Usa la CLI de Vault para acceder a los secretos almacenados:
    ```bash
    docker exec -it security vault kv get secret/myapp
    ```
- **Verificar estado**:
  - Comprueba el estado de Vault:
    ```bash
    curl http://localhost:8200/v1/sys/health
    ```

### **3. Pruebas con ModSecurity**
- **Simular un ataque**:
  - Realiza una solicitud HTTP maliciosa (por ejemplo, una inyección SQL) y verifica que ModSecurity la bloquee.
  - Ejemplo de solicitud:
    ```bash
    curl -X POST http://localhost:8080 --data "param=' OR '1'='1"
    ```
- **Revisar logs**:
  - Verifica los logs de ModSecurity en `/var/log/apache2/modsec_audit.log` dentro del contenedor.

### **4. Pruebas de integración**
- **Backend y Vault**:
  - Verifica que el backend pueda acceder a los secretos almacenados en Vault.
  - Ejemplo:
    ```bash
    curl http://localhost:3000/api/secret
    ```
- **Frontend y backend**:
  - Asegúrate de que el frontend se comunique correctamente con el backend y que los datos se muestren correctamente.

---

### **Resumen final**
- **OWASP ZAP**: Para escaneo de vulnerabilidades.
- **HashiCorp Vault**: Para gestión de secretos.
- **ModSecurity**: Para protección contra ataques web.
- **Apache HTTP Server**: Para servir la aplicación y integrar ModSecurity.

### **Próximos pasos**
1. Automatizar los escaneos de ZAP en tu pipeline de CI/CD.
2. Configurar Vault para un entorno de producción (modo no desarrollo).
3. Añadir más reglas a ModSecurity para mejorar la protección.
4. Realizar pruebas de carga y estrés para asegurar que la aplicación sea resistente.

¡Espero que este resumen te sea útil! Si tienes más preguntas o necesitas más detalles, no dudes en preguntar. 😊